# SWSAcademy - Cleanup and Billing

Run this notebook after a training session to:
1. **See what the session cost** (DBUs and estimated USD from the `system.billing` tables).
2. **Return the workspace to a clean slate** by removing everything the SWSAcademy courses created:
   - the `swsacademy` catalog (learner schemas, tables, feature tables, volumes, metadata)
   - the Marketplace dataset catalogs installed by the admin setup (`swsacademy_*`)
   - MLflow experiments and registered models created by the courses
   - Model Serving endpoints, Lakeflow pipelines and jobs created by the courses (these keep billing)
   - Lakehouse Monitoring dashboards and any saved `var/credentials.cfg` token files
   - temporary `_TEST ...` notebooks left behind by automated test runs

Nothing is deleted unless the **confirm_delete** widget is set to `YES`. With any other value the
notebook only lists what *would* be removed (dry run).

Re-run `00 - SWSAcademy Admin Setup` in a course to rebuild everything afterwards.

In [ ]:
%pip install -q -U databricks-sdk==0.36.0
%restart_python

In [ ]:
dbutils.widgets.text("confirm_delete", "NO", "Type YES to delete")
dbutils.widgets.text("billing_days", "7", "Billing lookback (days)")
dbutils.widgets.text("catalog_prefix", "swsacademy", "Catalog prefix to remove")
dbutils.widgets.text("courses_folder", "", "Courses folder (blank = auto)")

In [ ]:
import re
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
DELETE = dbutils.widgets.get("confirm_delete").strip().upper() == "YES"
DAYS = int(dbutils.widgets.get("billing_days") or 7)
PREFIX = dbutils.widgets.get("catalog_prefix").strip().lower()
me = w.current_user.me().user_name
courses_folder = dbutils.widgets.get("courses_folder").strip() or f"/Users/{me}/SWSAcademy Courses"

print(f"Mode: {'DELETE' if DELETE else 'DRY RUN (set confirm_delete = YES to delete)'}")
print(f"User: {me}\nCatalog prefix: {PREFIX}\nCourses folder: {courses_folder}")

## 1. Billing for the last N days

Cost is estimated from `system.billing.usage` joined with the list prices in
`system.billing.list_prices`. Billing data usually lags by a few hours, so run this again
later in the day for final numbers. Your actual invoice may differ if you have discounts.

In [ ]:
billing_sql = f"""
WITH usage AS (
  SELECT u.usage_date, u.sku_name, u.billing_origin_product AS product,
         u.usage_quantity AS dbus,
         u.usage_quantity * p.pricing.effective_list.default AS est_usd
  FROM system.billing.usage u
  LEFT JOIN system.billing.list_prices p
    ON u.sku_name = p.sku_name
   AND u.cloud = p.cloud
   AND u.usage_start_time >= p.price_start_time
   AND (p.price_end_time IS NULL OR u.usage_start_time < p.price_end_time)
  WHERE u.usage_date >= date_sub(current_date(), {DAYS})
)
SELECT usage_date, product, sku_name,
       ROUND(SUM(dbus), 3) AS dbus,
       ROUND(SUM(est_usd), 2) AS est_usd
FROM usage
GROUP BY ALL
ORDER BY usage_date DESC, est_usd DESC
"""
try:
    billing = spark.sql(billing_sql)
    display(billing)
    total = billing.selectExpr("ROUND(SUM(dbus), 3) AS dbus", "ROUND(SUM(est_usd), 2) AS usd").first()
    print(f"Last {DAYS} days: {total['dbus']} DBUs, approx. ${total['usd']} (list price, compute only;"
          f" Azure storage/network is billed separately by Azure)")
except Exception as e:
    print("Billing tables are not accessible for this user, so cost cannot be shown here.\n"
          f"  ({str(e).splitlines()[0][:200]})\n"
          "To enable: an account admin runs (once, in a SQL editor):\n"
          f"  GRANT USE SCHEMA ON SCHEMA system.billing TO `{me}`;\n"
          f"  GRANT SELECT ON SCHEMA system.billing TO `{me}`;\n"
          "Alternative: Azure portal > Cost Management > Cost analysis, filtered to this Databricks workspace.")

## 2. Find everything the courses created

In [ ]:
catalogs = sorted(c.name for c in w.catalogs.list() if c.name and c.name.lower().startswith(PREFIX))
installations = [i for i in w.consumer_installations.list()
                 if i.catalog_name and i.catalog_name.lower().startswith(PREFIX)]
installed_catalogs = {i.catalog_name for i in installations}

# Course notebooks create experiments directly under /Users/<you>/ with these name prefixes.
# Add new prefixes here as more courses are tested, instead of using a wildcard that could
# delete your own experiments.
COURSE_EXPERIMENT_PREFIXES = [
    "Demo-1.2-Model-Tracking-with-MLflow",
    "LAB-1-Model-Development-Tracking-with-MLflow",
    "Hyperparameter_Tuning_with_Optuna_",
    "Lab_Optuna_Experiment_",
    "loan_model_comparison",
    "churn_prediction_genie",
    "4_1_Real-Time-Deployment",
    "Lab-1.2b-Model-traning-with-MLflow",
    "Lab-2.1-Model-traning-with-MLflow",
]
user_root = f"/Users/{me}/"
instructor_folder = f"/Users/{me}/SWSAcademy Instructor Demos"   # copy used by the SWSAcademy demo jobs
experiments = [
    e for e in w.experiments.list_experiments()
    if e.name and (
        e.name.startswith(courses_folder)
        or e.name.startswith(instructor_folder)
        or PREFIX in e.name.lower()
        or any(e.name.startswith(user_root + p) for p in COURSE_EXPERIMENT_PREFIXES)
    )
]

models = []
for cat in catalogs:
    if cat in installed_catalogs:
        continue  # Marketplace shares are read-only
    for sch in w.schemas.list(catalog_name=cat):
        if sch.name == "information_schema":
            continue
        try:
            models += list(w.registered_models.list(catalog_name=cat, schema_name=sch.name))
        except Exception as e:
            print(f"Could not list models in {cat}.{sch.name}: {e}")

def _exists(path):
    try:
        w.workspace.get_status(path)
        return True
    except Exception:
        return False

def find_test_notebooks(path):
    found = []
    try:
        for obj in w.workspace.list(path):
            if obj.object_type.value == "DIRECTORY":
                found += find_test_notebooks(obj.path)
            elif obj.path.rsplit("/", 1)[-1].startswith("_TEST ") or obj.path.endswith("/var/credentials.cfg"):
                found.append(obj.path)
    except Exception:
        pass
    return found

test_notebooks = find_test_notebooks(courses_folder)

# Serving endpoints: course demos tag them "sws_academy" (older runs: "db_academy") and name them "ML_AS_..."
endpoints = []
for ep in w.serving_endpoints.list():
    tags = {t.key for t in (ep.tags or [])}
    if {"db_academy", "sws_academy"} & tags or "ML_AS_" in (ep.name or ""):
        endpoints.append(ep.name)

# Jobs created by the MLOps demos (via UI, REST API and CLI)
JOB_NAME_MARKERS = ["-testing-workflow-job", "MLOps Workflow"]
jobs = [(j.job_id, j.settings.name) for j in w.jobs.list()
        if j.settings and j.settings.name and any(m in j.settings.name for m in JOB_NAME_MARKERS)]

# Workspace files created by the courses outside the courses folder
extra_paths = [p for p in (f"/Users/{me}/databricks_lakehouse_monitoring",)
               if _exists(p)]

# Pipelines that write into a course catalog
pipelines = []
for p in w.pipelines.list_pipelines():
    try:
        spec = w.pipelines.get(p.pipeline_id).spec
        # Keep instructor assets ("SWSAcademy - ..." pipelines used by the demo jobs)
        if (p.name or "").startswith("SWSAcademy - "):
            continue
        if spec and (spec.catalog or "").lower().startswith(PREFIX):
            pipelines.append((p.pipeline_id, p.name))
    except Exception as e:
        print(f"Could not inspect pipeline {p.name}: {e}")

print("Catalogs:              ", catalogs or "-")
print("  of which Marketplace:", sorted(installed_catalogs) or "-")
print("Registered models:     ", [m.full_name for m in models] or "-")
print("MLflow experiments:    ", [e.name for e in experiments] or "-")
print("_TEST notebooks / saved tokens:", test_notebooks or "-")
print("Jobs:                  ", [n for _, n in jobs] or "-")
print("Other workspace files: ", extra_paths or "-")
print("Serving endpoints:     ", endpoints or "-")
print("Pipelines:             ", [n for _, n in pipelines] or "-")

## 3. Delete (only when confirm_delete = YES)

In [ ]:
if not DELETE:
    print("Dry run only. Nothing was deleted. Set confirm_delete = YES and re-run this cell.")
else:
    # Endpoints first: they serve model versions that are deleted below.
    for name in endpoints:
        try:
            w.serving_endpoints.delete(name)
            print("Deleted serving endpoint", name)
        except Exception as e:
            print("Endpoint", name, "->", e)

    for jid, name in jobs:
        try:
            w.jobs.delete(jid)
            print("Deleted job", name)
        except Exception as e:
            print("Job", name, "->", e)

    for p in extra_paths:
        try:
            w.workspace.delete(p, recursive=True)
            print("Deleted", p)
        except Exception as e:
            print("Delete", p, "->", e)

    for pid, name in pipelines:
        try:
            w.pipelines.delete(pid)
            print("Deleted pipeline", name)
        except Exception as e:
            print("Pipeline", name, "->", e)

    for m in models:
        try:
            for v in w.model_versions.list(m.full_name):
                w.model_versions.delete(m.full_name, v.version)
            w.registered_models.delete(m.full_name)
            print("Deleted model", m.full_name)
        except Exception as e:
            print("Model", m.full_name, "->", e)

    def is_notebook_experiment(name):
        # A notebook's own experiment has the notebook's path as its name. Deleting it is permanent and leaves the
        # notebook unable to log runs, so these are only emptied, never deleted.
        try:
            return w.workspace.get_status(name).object_type.value == "NOTEBOOK"
        except Exception:
            return False

    for e in experiments:
        try:
            if is_notebook_experiment(e.name):
                raise RuntimeError("notebook experiment")
            w.experiments.delete_experiment(e.experiment_id)
            print("Deleted experiment", e.name)
        except Exception as ex:
            # Notebook experiments belong to their notebook and cannot be deleted;
            # remove their runs instead so they are left empty.
            try:
                run_ids = [r.info.run_id for r in w.experiments.search_runs(experiment_ids=[e.experiment_id])]
                for run_id in run_ids:
                    w.experiments.delete_run(run_id)
                print(f"Emptied notebook experiment ({len(run_ids)} runs):", e.name.rsplit("/", 1)[-1])
            except Exception as ex2:
                print("Experiment", e.name, "->", ex, "|", ex2)

    for inst in installations:
        try:
            w.consumer_installations.delete(inst.listing_id, inst.id)
            print("Uninstalled Marketplace listing ->", inst.catalog_name)
        except Exception as e:
            print("Uninstall", inst.catalog_name, "->", e)

    for cat in catalogs:
        try:
            spark.sql(f"DROP CATALOG IF EXISTS `{cat}` CASCADE")
            print("Dropped catalog", cat)
        except Exception as e:
            print("Drop", cat, "->", e)

    for nb in test_notebooks:
        w.workspace.delete(nb)
        print("Deleted", nb)

    left = [c.name for c in w.catalogs.list() if c.name and c.name.lower().startswith(PREFIX)]
    print("\nRemaining catalogs with prefix:", left or "none - clean slate")

## 4. Make sure nothing keeps billing

Serverless notebooks and jobs stop billing on their own when idle. This cell stops any
running SQL warehouse or classic cluster so nothing keeps running overnight.

In [ ]:
for wh in w.warehouses.list():
    if wh.state and wh.state.value in ("RUNNING", "STARTING"):
        print("Running warehouse:", wh.name)
        if DELETE:
            w.warehouses.stop(wh.id)
            print("  -> stop requested")

for cl in w.clusters.list():
    if cl.state and cl.state.value in ("RUNNING", "PENDING", "RESIZING"):
        print("Running cluster:", cl.cluster_name)
        if DELETE:
            w.clusters.delete(cl.cluster_id)
            print("  -> terminate requested")

print("Done.")